# Interactive Denoising Lab — PandaOmron

Experiment with GR00T N1.6's flow-matching denoising loop using
`ROBOCASA_PANDA_OMRON` observations captured from the simulator.

PandaOmron uses **EEF delta** actions, so the 3D plots show true Cartesian
end-effector trajectories.

## Prerequisites

Before running this notebook, capture at least one observation `.npz` file
using `interactive_rollout.py`:

```bash
# Terminal 1 (model venv)
uv run python gr00t/eval/run_gr00t_server.py \
  --model-path nvidia/GR00T-N1.6-3B \
  --embodiment-tag ROBOCASA_PANDA_OMRON \
  --use-sim-policy-wrapper

# Terminal 2 (sim venv)
gr00t/eval/sim/robocasa/robocasa_uv/.venv/bin/python \
  scripts/denoising_lab/interactive_rollout.py \
  --env-name robocasa_panda_omron/OpenDrawer_PandaOmron_Env \
  --host 127.0.0.1 --port 5555 \
  --n-action-steps 8 --max-episode-steps 720 \
  --save-dir /tmp/saved_observations
```

Then press `o` in the interactive rollout to save an observation, and set
`OBS_PATH` below to the saved `.npz` file.

In [ ]:
# Cell 1: Imports + config
import sys, os
import numpy as np
import torch
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from scripts.denoising_lab.denoising_lab import (
    DenoisingLab, TrajectoryVisualizer, compare_strategies,
)

MODEL_PATH = "nvidia/GR00T-N1.6-3B"
EMBODIMENT_TAG = "robocasa_panda_omron"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Path to observation .npz captured by interactive_rollout.py
OBS_PATH = "/tmp/saved_observations/ep000_step000.npz"

print(f"Device: {DEVICE}")

In [ ]:
# Cell 2: Load model
lab = DenoisingLab(MODEL_PATH, EMBODIMENT_TAG, device=DEVICE)
print(f"Model loaded. Action horizon: {lab.action_horizon}, Action dim: {lab.action_dim}")
print(f"Inference timesteps: {lab.num_inference_timesteps}, Timestep buckets: {lab.num_timestep_buckets}")

In [ ]:
# Cell 3: Load observation from .npz saved by interactive_rollout.py
obs = DenoisingLab.load_observation(OBS_PATH)
print("Observation loaded from:", OBS_PATH)
print("Keys:", list(obs.keys()))
for k, v in obs.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# Cell 4: Encode backbone features (EXPENSIVE — run once)
features = lab.encode_features_from_sim_obs(obs)
print(f"Backbone features: {features.backbone_features.shape}")
print(f"State features: {features.state_features.shape}")
print(f"Embodiment ID: {features.embodiment_id}")

In [ ]:
# Cell 5: Default denoising (4-step flow matching)
result = lab.denoise(features, seed=42)
print(f"Final action shape: {result.action_pred.shape}")
print(f"Seed: {result.seed}")
for info in result.intermediates:
    print(f"  Step {info.step}: t_cont={info.t_cont:.3f} t_disc={info.t_discretized} "
          f"action_norm={info.action_norm:.4f} velocity_norm={info.velocity_norm:.4f}")

In [ ]:
# Cell 6: Decode + inspect actions
decoded = lab.decode_raw_actions(result.action_pred, features.states)
print("Decoded action keys (PandaOmron EEF):")
for key, arr in decoded.items():
    print(f"  {key}: shape={arr.shape}")

for t in range(min(3, list(decoded.values())[0].shape[1])):
    print(lab.label_action_step(decoded, t))

In [ ]:
# Cell 7: 3D EEF trajectory plot
EEF_KEY = "end_effector_position"
print(f"Plotting EEF trajectory using key: '{EEF_KEY}'")

viz = TrajectoryVisualizer()
viz.add_trajectory(decoded, "Default (4-step, seed=42)", eef_key=EEF_KEY)
fig = viz.plot_eef_3d(title="Default Denoising — EEF Trajectory")
plt.show()

In [ ]:
# Cell 8: Compare seeds — 5 different seeds on the same 3D plot
viz_seeds = TrajectoryVisualizer()
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

for i, seed in enumerate([0, 42, 123, 456, 789]):
    r = lab.denoise(features, seed=seed)
    d = lab.decode_raw_actions(r.action_pred, features.states)
    viz_seeds.add_trajectory(d, f"seed={seed}", color=colors[i], eef_key=EEF_KEY)

fig = viz_seeds.plot_eef_3d(title="Seed Comparison — Same Observation, Different Noise")
plt.show()

fig2 = viz_seeds.plot_eef_components(title="Seed Comparison — EEF Components")
plt.show()

In [ ]:
# Cell 9: Compare step counts — 2/4/8/16 denoising steps
strategies = [
    {"num_steps": 2, "seed": 42},
    {"num_steps": 4, "seed": 42},
    {"num_steps": 8, "seed": 42},
    {"num_steps": 16, "seed": 42},
]
labels = ["2 steps", "4 steps", "8 steps", "16 steps"]

results, viz_steps = compare_strategies(lab, features, strategies, labels, eef_key=EEF_KEY)

fig = viz_steps.plot_eef_3d(title="Step Count Comparison (same seed=42)")
plt.show()

fig2 = viz_steps.plot_eef_components(title="Step Count Comparison — EEF Components")
plt.show()

In [ ]:
# Cell 10: Manual step-by-step denoising
gen = torch.Generator(device=lab.device).manual_seed(42)
actions = torch.randn(
    (1, lab.action_horizon, lab.action_dim),
    dtype=lab.dtype, device=lab.device, generator=gen,
)

num_steps = 4
print(f"Starting manual {num_steps}-step denoising...")
print(f"Initial noise norm: {actions.float().norm().item():.4f}")

for step in range(num_steps):
    velocity, actions = lab.denoise_single_step(features, actions, step, num_steps)
    print(f"  Step {step}: action_norm={actions.float().norm().item():.4f} "
          f"velocity_norm={velocity.float().norm().item():.4f}")

manual_decoded = lab.decode_raw_actions(actions, features.states)
print("\nManual denoising complete. Decoded keys:", list(manual_decoded.keys()))

In [ ]:
# Cell 11: Guided denoising — scale velocity by 1.5x
from functools import partial

def scale_velocity(actions, step_idx, velocity, scale=1.5):
    return velocity * scale

result_guided = lab.denoise(features, seed=42, guided_fn=partial(scale_velocity, scale=1.5))
result_normal = lab.denoise(features, seed=42)

viz_guide = TrajectoryVisualizer()
d_normal = lab.decode_raw_actions(result_normal.action_pred, features.states)
d_guided = lab.decode_raw_actions(result_guided.action_pred, features.states)

viz_guide.add_trajectory(d_normal, "Normal", color="tab:blue", eef_key=EEF_KEY)
viz_guide.add_trajectory(d_guided, "Guided (1.5x velocity)", color="tab:red", eef_key=EEF_KEY)

fig = viz_guide.plot_eef_3d(title="Guided vs Normal Denoising — EEF Trajectory")
plt.show()

In [ ]:
# Cell 12: Raw denoising loop — full playground mode
seed = 42
num_steps = 4
dt = 1.0 / num_steps

vl_embeds = features.backbone_features
state_feats = features.state_features
emb_id = features.embodiment_id
bb_output = features.backbone_output
B = vl_embeds.shape[0]
device = vl_embeds.device

gen = torch.Generator(device=device).manual_seed(seed)
actions = torch.randn(
    (B, lab.action_horizon, lab.action_dim),
    dtype=vl_embeds.dtype, device=device, generator=gen,
)

for t in range(num_steps):
    t_cont = t / float(num_steps)
    t_disc = int(t_cont * lab.num_timestep_buckets)

    ts = torch.full((B,), t_disc, device=device)
    act_feat = lab.action_head.action_encoder(actions, ts, emb_id)

    if lab.action_head.config.add_pos_embed:
        pos = torch.arange(act_feat.shape[1], dtype=torch.long, device=device)
        act_feat = act_feat + lab.action_head.position_embedding(pos).unsqueeze(0)

    sa = torch.cat((state_feats, act_feat), dim=1)

    if lab.action_head.config.use_alternate_vl_dit:
        out = lab.action_head.model(
            hidden_states=sa, encoder_hidden_states=vl_embeds, timestep=ts,
            image_mask=bb_output.image_mask,
            backbone_attention_mask=bb_output.backbone_attention_mask,
        )
    else:
        out = lab.action_head.model(
            hidden_states=sa, encoder_hidden_states=vl_embeds, timestep=ts,
        )

    pred = lab.action_head.action_decoder(out, emb_id)
    velocity = pred[:, -lab.action_horizon:]

    # === EDIT HERE: modify velocity, inject guidance, etc. ===
    # velocity = velocity * 1.0  # no-op example

    actions = actions + dt * velocity
    print(f"Step {t}: action_norm={actions.float().norm():.4f} vel_norm={velocity.float().norm():.4f}")

raw_decoded = lab.decode_raw_actions(actions, features.states)
print("\nDone. Decoded keys:", list(raw_decoded.keys()))

In [ ]:
# Cell 13: Denoising progression visualization
result_for_prog = lab.denoise(features, seed=42)
fig = viz.plot_denoising_progression(result_for_prog, lab, eef_key=EEF_KEY)
plt.show()